# 03 · Baselines (Table 6) & ablation (Table 5)
VGG16 / VGG19 / InceptionV3 (ImageNet weights, fully fine-tuned, GAP + Dense256 + Dropout.5 head) under exactly the same CV protocol, then the architectural ablations of FireNet-CNN.

⚠ Baselines are the slow part (5 folds × 50 epochs × 3 models). Keep `EPOCHS=2` for a smoke test; for the real run expect a few hours on a T4 - Colab may disconnect, but finished runs are saved to Drive (`runs/<dataset>_<mode>_<tag>/summary.json`), so re-run and skip what's done (the `skip_done` helper below).

In [ ]:
# ---- Colab setup (run first; if you ever see Keras-3 errors: Runtime > Restart session, then re-run) ----
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # the paper's code style needs Keras 2
!pip -q install tf_keras
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/firenet"       # everything lives here, so a disconnect loses nothing
sys.path.insert(0, PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from types import SimpleNamespace
from IPython.display import Image, display
import firenet as fn                             # written to Drive by 00_setup_and_data.ipynb
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
DATASETS = ["A", "B"]          # which real dataset(s)
MODES    = ["paper", "strict"] # paper = split AFTER adding synthetic (as in the paper); strict = synthetic only in training folds
EPOCHS   = 2                   # <-- smoke test first! then set to 50 for the real run

cfg = SimpleNamespace(data=f"{PROJECT}/data", dataset="B", mode="paper", no_synth=False,
                      epochs=EPOCHS, bs=32, lr=1e-4, seed=42, out=f"{PROJECT}/runs")
def get(d):
    return fn.load_cached(cfg.data, d, use_synth=True, cache_dir=f"{PROJECT}/cache")

In [ ]:
def skip_done(d, mode, tag):
    return os.path.exists(f"{PROJECT}/runs/{d}_{mode}_{tag}/summary.json")

## Baselines

In [ ]:
for d in DATASETS:
    cfg.dataset = d; X, y, s = get(d)
    for mode in MODES:
        cfg.mode = mode
        for name in ("vgg16", "vgg19", "inceptionv3"):
            if skip_done(d, mode, name): print("skip", d, mode, name); continue
            fn.run_cv(X, y, s, lambda n=name: fn.build_pretrained(n), cfg, name)

## Ablation (Dataset B, as in Table 5)

In [ ]:
VARIANTS = {"original": ({}, {}), "no_conv": ({"conv": False}, {}), "leakyrelu": ({"act": "leaky_relu"}, {}),
            "sigmoid_act": ({"act": "sigmoid"}, {}), "no_dropout": ({"dropout": False}, {}), "no_bn": ({"bn": False}, {}),
            "sgd": ({}, {"opt": "sgd"}), "rmsprop": ({}, {"opt": "rmsprop"}), "mse_loss": ({}, {"loss": "mse"})}
ABL_DATASET, ABL_MODE = "B", "paper"
cfg.dataset, cfg.mode = ABL_DATASET, ABL_MODE
X, y, s = get(ABL_DATASET)
for name, (bkw, ckw) in VARIANTS.items():
    tag = f"abl_{name}"
    if skip_done(ABL_DATASET, ABL_MODE, tag): print("skip", tag); continue
    fn.run_cv(X, y, s, lambda b=bkw: fn.build_firenet(**b), cfg, tag, ckw)

In [ ]:
rows = {}
for name in VARIANTS:
    p = f"{PROJECT}/runs/{ABL_DATASET}_{ABL_MODE}_abl_{name}/summary.json"
    if os.path.exists(p): rows[name] = pd.read_json(p, typ="series")
abl = (pd.DataFrame(rows).T[["train_acc", "accuracy", "precision", "f1"]] * 100).round(2)
abl.columns = ["Train Acc (%)", "Val Acc (%)", "Precision (%)", "F1 (%)"]
abl.to_csv(f"{PROJECT}/runs/ablation_{ABL_DATASET}_{ABL_MODE}.csv"); abl